In [2]:
import pandas as pd
import numpy as np

In [3]:
retail = pd.read_csv("../data/raw/retail_price.csv")
online = pd.read_csv("../data/raw/online_retail_II.csv")

In [4]:
print("Retail:", retail.shape)
print("Online:", online.shape)

Retail: (676, 30)
Online: (1067371, 8)


In [5]:
retail["month_year"] = pd.to_datetime(
    retail["month_year"],
    format="%d-%m-%Y"
)

In [6]:
retail.isnull().sum()

product_id                    0
product_category_name         0
month_year                    0
qty                           0
total_price                   0
freight_price                 0
unit_price                    0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_score                 0
customers                     0
weekday                       0
weekend                       0
holiday                       0
month                         0
year                          0
s                             0
volume                        0
comp_1                        0
ps1                           0
fp1                           0
comp_2                        0
ps2                           0
fp2                           0
comp_3                        0
ps3                           0
fp3                           0
lag_price                     0
dtype: int64

In [7]:
retail = retail.sort_values(
    ["product_id", "month_year"]
).reset_index(drop=True)

In [8]:
print(retail["month_year"].min())
print(retail["month_year"].max())

2017-01-01 00:00:00
2018-08-01 00:00:00


In [9]:
retail["calculated_total"] = (
    retail["qty"] * retail["unit_price"]
)

In [10]:
retail["price_difference"] = (
    retail["total_price"] - retail["calculated_total"]
)

print(retail["price_difference"].describe())

count    676.000000
mean       0.488275
std        8.115285
min      -44.290909
25%        0.000000
50%        0.000000
75%        0.000000
max      147.551064
Name: price_difference, dtype: float64


In [11]:
print(
    retail[
        retail["price_difference"].abs() > 0.01
    ][
        ["qty", "unit_price", "total_price",
         "calculated_total", "price_difference"]
    ].head(20)
)

    qty  unit_price  total_price  calculated_total  price_difference
26   18   88.488235      1594.20       1592.788235          1.411765
33   10   77.933333       781.30        779.333333          1.966667
38   11   98.300000      1082.00       1081.300000          0.700000
41    9   90.875000       815.00        817.875000         -2.875000
47   32   44.154444      1421.67       1412.942222          8.727778
59   52  199.509804     10375.00      10374.509803          0.490197
60   32  163.398710      5222.36       5228.758710         -6.398710
62    6  100.900000       611.40        605.400000          6.000000
69   31   90.534000      2795.14       2806.554000        -11.414000
70   30   84.756087      2537.19       2542.682609         -5.492609
71   13   87.500000      1139.80       1137.500000          2.300000
74   18   66.342143      1199.48       1194.158571          5.321429
78   29   97.642500      2831.55       2831.632500         -0.082500
79   71   78.712281      5581.80  

In [12]:
print("Negative quantities:",
      (online["Quantity"] < 0).sum())

print("Zero quantities:",
      (online["Quantity"] == 0).sum())

print("Negative prices:",
      (online["Price"] < 0).sum())

print("Zero prices:",
      (online["Price"] == 0).sum())

Negative quantities: 22950
Zero quantities: 0
Negative prices: 5
Zero prices: 6202


In [13]:
print(
    online[online["Quantity"] < 0]
    [["Invoice", "StockCode", "Description",
      "Quantity", "InvoiceDate", "Price"]]
    .head(20)
)

     Invoice StockCode                          Description  Quantity  \
178  C489449     22087             PAPER BUNTING WHITE LACE       -12   
179  C489449    85206A         CREAM FELT EASTER EGG BASKET        -6   
180  C489449     21895        POTTING SHED SOW 'N' GROW SET        -4   
181  C489449     21896                   POTTING SHED TWINE        -6   
182  C489449     22083           PAPER CHAIN KIT RETRO SPOT       -12   
183  C489449     21871                  SAVE THE PLANET MUG       -12   
184  C489449     84946      ANTIQUE SILVER TEA GLASS ETCHED       -12   
185  C489449    84970S    HANGING HEART ZINC T-LIGHT HOLDER       -24   
186  C489449     22090            PAPER BUNTING RETRO SPOTS       -12   
196  C489459    90200A           PURPLE SWEETHEART BRACELET        -3   
197  C489459    90200D             PINK SWEETHEART BRACELET        -3   
198  C489459    90200B            BLACK SWEETHEART BRACELET        -3   
199  C489459    90200E            GREEN SWEETHEART 

In [14]:
print(
    online[online["Price"] <= 0]
    [["Invoice", "StockCode", "Description",
      "Quantity", "InvoiceDate", "Price"]]
    .head(20)
)

     Invoice StockCode          Description  Quantity          InvoiceDate  \
263   489464     21733         85123a mixed       -96  2009-12-01 10:52:00   
283   489463     71477                short      -240  2009-12-01 10:52:00   
284   489467    85123A          21733 mixed      -192  2009-12-01 10:53:00   
470   489521     21646                  NaN       -50  2009-12-01 11:44:00   
3114  489655     20683                  NaN       -44  2009-12-01 17:26:00   
3161  489659     21350                  NaN       230  2009-12-01 17:39:00   
3162  489660     35956                 lost     -1043  2009-12-01 17:43:00   
3168  489663    35605A              damages      -117  2009-12-01 18:02:00   
3731  489781     84292                  NaN        17  2009-12-02 11:45:00   
4296  489806     18010                  NaN      -770  2009-12-02 12:42:00   
4538  489820     21133      invcd as 84879?      -720  2009-12-02 13:23:00   
4566  489821    85049G                  NaN      -240  2009-12-0

In [15]:
online["InvoiceDate"] = pd.to_datetime(
    online["InvoiceDate"]
)

In [16]:
print(online["InvoiceDate"].min())
print(online["InvoiceDate"].max())

2009-12-01 07:45:00
2011-12-09 12:50:00


In [17]:
# Check cancellation invoices

cancelled_invoices = online["Invoice"].astype(str).str.startswith("C")

print("Cancellation invoices:", cancelled_invoices.sum())
print("Total transactions:", len(online))

Cancellation invoices: 19494
Total transactions: 1067371


In [18]:
# Compare negative quantities with cancellation invoices

print(
    pd.crosstab(
        cancelled_invoices,
        online["Quantity"] < 0
    )
)

Quantity    False  True 
Invoice                 
False     1044420   3457
True            1  19493


In [19]:
negative_price = online[online["Price"] < 0]

print("Negative price records:")
display(
    negative_price[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ]
)

Negative price records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06


In [20]:
zero_price = online[online["Price"] == 0]

print("Zero-price transactions:", len(zero_price))

display(
    zero_price[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ].head(30)
)

Zero-price transactions: 6202


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0


In [21]:
online["is_cancelled"] = (
    online["Invoice"]
    .astype(str)
    .str.startswith("C")
)

print(
    online.groupby("is_cancelled")["Quantity"]
    .agg(["count", "min", "max", "mean"])
)

                count    min    max       mean
is_cancelled                                  
False         1047877  -9600  80995  10.592354
True            19494 -80995      1 -25.186827


In [22]:
non_cancelled_negative = online[
    (~online["is_cancelled"]) &
    (online["Quantity"] < 0)
]

print("Negative quantity but non-cancellation:",
      len(non_cancelled_negative))

display(
    non_cancelled_negative[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price"]
    ].head(50)
)

Negative quantity but non-cancellation: 3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0


In [23]:
zero_price_summary = (
    online[online["Price"] == 0]
    ["Description"]
    .value_counts(dropna=False)
    .head(30)
)

display(zero_price_summary)

Description
NaN                                    4382
check                                   162
?                                        92
damages                                  84
damaged                                  81
found                                    28
missing                                  27
sold as set on dotcom                    20
Damaged                                  17
adjustment                               16
OWL DOORSTOP                             15
POLYESTER FILLER PAD 45x45cm             12
dotcom                                   12
amazon                                   11
FRENCH BLUE METAL DOOR SIGN 1            10
IVORY KITCHEN SCALES                     10
POLYESTER FILLER PAD 40x40cm             10
AIRLINE BAG VINTAGE WORLD CHAMPION        9
smashed                                   9
RED KITCHEN SCALES                        9
BOX OF 24 COCKTAIL PARASOLS               9
FRENCH BLUE METAL DOOR SIGN 8             9
Unsaleable, destroye

In [24]:
print("Missing Customer ID:",
      online["Customer ID"].isnull().sum())

print("Missing Customer ID percentage:",
      online["Customer ID"].isnull().mean() * 100)

Missing Customer ID: 243007
Missing Customer ID percentage: 22.766872999172733


In [25]:
print("Missing Description:",
      online["Description"].isnull().sum())

print("Missing Description percentage:",
      online["Description"].isnull().mean() * 100)

Missing Description: 4382
Missing Description percentage: 0.4105414143723223


In [26]:
online_clean = online.copy()
print("Original rows:", len(online))

Original rows: 1067371


In [27]:
online_clean = online_clean[
    online_clean["is_cancelled"] == False
]

print("After removing cancellations:", len(online_clean))

After removing cancellations: 1047877


In [28]:
online_clean = online_clean[
    online_clean["Quantity"] > 0
]

print("After removing non-positive quantities:", len(online_clean))

After removing non-positive quantities: 1044420


In [30]:
online_clean = online_clean[
    online_clean["Price"] > 0
]

print("After removing non-positive prices:", len(online_clean))

After removing non-positive prices: 1041670


In [34]:
print("Final shape:", online_clean.shape)

print("Negative quantities:",
      (online_clean["Quantity"] < 0).sum())

print("Zero quantities:",
      (online_clean["Quantity"] == 0).sum())

print("Negative prices:",
      (online_clean["Price"] < 0).sum())

print("Zero prices:",
      (online_clean["Price"] == 0).sum())

Final shape: (1041670, 8)
Negative quantities: 0
Zero quantities: 0
Negative prices: 0
Zero prices: 0


In [35]:
# Check duplicate rows
duplicate_count = online_clean.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 33757


In [36]:
duplicate_count = online_clean.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 33757


In [37]:
# Check some duplicate rows
duplicates = online_clean[
    online_clean.duplicated(keep=False)
].sort_values(
    ["Invoice", "StockCode", "InvoiceDate"]
)

print("Total rows involved in duplicates:", len(duplicates))


Total rows involved in duplicates: 66094


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [38]:
display(duplicates.head(30))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [39]:
# Remove exact duplicate transactions
before_duplicates = len(online_clean)

online_clean = online_clean.drop_duplicates()

after_duplicates = len(online_clean)

print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates:", after_duplicates)
print("Duplicate rows removed:", before_duplicates - after_duplicates)

Rows before removing duplicates: 1041670
Rows after removing duplicates: 1007913
Duplicate rows removed: 33757


In [40]:
print("Remaining duplicate rows:",
      online_clean.duplicated().sum())

Remaining duplicate rows: 0


In [41]:
# Check missing values in cleaned dataset

missing_values = online_clean.isnull().sum()

print(missing_values)

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228488
Country             0
dtype: int64


In [42]:
# Show missing-value percentages

missing_percentage = (
    online_clean.isnull().mean() * 100
).round(2)

print(missing_percentage)

Invoice         0.00
StockCode       0.00
Description     0.00
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.67
Country         0.00
dtype: float64


In [43]:
# Check data types
print(online_clean.dtypes)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object


In [44]:
# Check basic dataset information
online_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1007913 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007913 non-null  object        
 1   StockCode    1007913 non-null  object        
 2   Description  1007913 non-null  object        
 3   Quantity     1007913 non-null  int64         
 4   InvoiceDate  1007913 non-null  datetime64[ns]
 5   Price        1007913 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007913 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 69.2+ MB


In [46]:
# Check Quantity and Price distribution
print("Quantity statistics:")
print(online_clean["Quantity"].describe())

print("\nPrice statistics:")
print(online_clean["Price"].describe())

Quantity statistics:
count    1.007913e+06
mean     1.111718e+01
std      1.284700e+02
min      1.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.200000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

Price statistics:
count    1.007913e+06
mean     4.074252e+00
std      5.043045e+01
min      1.000000e-03
25%      1.250000e+00
50%      2.100000e+00
75%      4.130000e+00
max      2.511109e+04
Name: Price, dtype: float64


In [47]:
# Check transactions with very high quantities

high_quantity = online_clean.nlargest(
    20, "Quantity"
)

display(
    high_quantity[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price",
         "Customer ID", "Country"]
    ]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom


In [48]:
# Check transactions with very high prices

high_price = online_clean.nlargest(
    20, "Price"
)

display(
    high_price[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price",
         "Customer ID", "Country"]
    ]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
241827,512771,M,Manual,1,2010-06-17 16:53:00,25111.09,NaN,United Kingdom
517955,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918.0,United Kingdom
135015,502265,M,Manual,1,2010-03-23 15:28:00,10953.50,NaN,United Kingdom
342147,522796,M,Manual,1,2010-09-16 15:12:00,10468.80,NaN,United Kingdom
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063.0,United Kingdom
372834,525399,M,Manual,1,2010-10-05 11:49:00,10468.80,NaN,United Kingdom
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949.0,United Kingdom
698843,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom


In [49]:
# Check special/non-product transactions

special_codes = [
    "M",
    "AMAZONFEE",
    "POST",
    "DOT",
    "ADJUST"
]

special_transactions = online_clean[
    online_clean["StockCode"].astype(str).isin(special_codes)
]

print("Special transactions:", len(special_transactions))

display(
    special_transactions[
        ["Invoice", "StockCode", "Description",
         "Quantity", "InvoiceDate", "Price",
         "Customer ID", "Country"]
    ].head(50)
)

Special transactions: 4151


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
89,489439,POST,POSTAGE,3,2009-12-01 09:28:00,18.00,12682.0,France
126,489444,POST,POSTAGE,1,2009-12-01 09:55:00,141.00,12636.0,USA
173,489447,POST,POSTAGE,1,2009-12-01 10:10:00,130.00,12362.0,Belgium
625,489526,POST,POSTAGE,6,2009-12-01 11:50:00,18.00,12533.0,Germany
1244,489557,POST,POSTAGE,4,2009-12-01 12:52:00,18.00,12490.0,France
2379,489597,DOT,DOTCOM POSTAGE,1,2009-12-01 14:28:00,647.19,NaN,United Kingdom
2539,489600,DOT,DOTCOM POSTAGE,1,2009-12-01 14:43:00,55.96,NaN,United Kingdom
2551,489601,DOT,DOTCOM POSTAGE,1,2009-12-01 14:44:00,68.39,NaN,United Kingdom
2571,489602,DOT,DOTCOM POSTAGE,1,2009-12-01 14:45:00,59.35,NaN,United Kingdom
2619,489603,DOT,DOTCOM POSTAGE,1,2009-12-01 14:46:00,42.39,NaN,United Kingdom


In [50]:
# Remove non-product transactions

special_codes = [
    "M",
    "AMAZONFEE",
    "POST",
    "DOT",
    "ADJUST"
]

before_special = len(online_clean)

online_clean = online_clean[
    ~online_clean["StockCode"].astype(str).isin(special_codes)
]

after_special = len(online_clean)

print("Rows before removing special transactions:", before_special)
print("Rows after removing special transactions:", after_special)
print("Special transactions removed:",
      before_special - after_special)

Rows before removing special transactions: 1007913
Rows after removing special transactions: 1003762
Special transactions removed: 4151


In [51]:
# Final cleaning validation

print("Final dataset shape:", online_clean.shape)

print("\nDuplicate rows:",
      online_clean.duplicated().sum())

print("\nMissing values:")
print(online_clean.isnull().sum())

print("\nQuantity:")
print("Minimum:", online_clean["Quantity"].min())
print("Negative:", (online_clean["Quantity"] < 0).sum())
print("Zero:", (online_clean["Quantity"] == 0).sum())

print("\nPrice:")
print("Minimum:", online_clean["Price"].min())
print("Negative:", (online_clean["Price"] < 0).sum())
print("Zero:", (online_clean["Price"] == 0).sum())

Final dataset shape: (1003762, 8)

Duplicate rows: 0

Missing values:
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    226869
Country             0
dtype: int64

Quantity:
Minimum: 1
Negative: 0
Zero: 0

Price:
Minimum: 0.001
Negative: 0
Zero: 0


In [52]:
online_clean.to_csv(
    "../data/processed/online_retail_clean.csv",
    index=False
)